<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.

Dans ce projet final, nous allons prédire si le premier étage du Falcon 9 atterrira avec succès. SpaceX annonce sur son site web un coût de 62 millions de dollars pour les lancements de fusées Falcon 9 ; d'autres fournisseurs proposent des lancements à partir de 165 millions de dollars chacun, une économie importante étant due à la réutilisation du premier étage par SpaceX. Par conséquent, si nous pouvons déterminer si le premier étage atterrira avec succès, nous pourrons calculer le coût d'un lancement. Cette information pourra être utilisée par une entreprise concurrente souhaitant proposer un lancement face à SpaceX. Dans ce TP, vous collecterez les données issues d'une API et vous assurerez de leur format correct. Voici un exemple de lancement réussi.

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [21]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.

Nous allons définir ci-dessous une série de fonctions auxiliaires qui nous permettront d'utiliser l'API pour extraire des informations à partir des numéros d'identification contenus dans les données de lancement.

Dans la colonne relative à la fusée, nous souhaitons obtenir le nom du propulseur.

In [22]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.

Depuis la rampe de lancement, nous aimerions connaître le nom du site de lancement utilisé, sa longitude et sa latitude.

In [23]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
         response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
         Longitude.append(response['longitude'])
         Latitude.append(response['latitude'])
         LaunchSite.append(response['name'])

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.

Nous aimerions connaître la masse de la charge utile et l'orbite qu'elle va atteindre.

In [24]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.

À partir des noyaux, nous aimerions connaître le résultat de l'atterrissage, le type d'atterrissage, le nombre de vols effectués avec ce noyau, si des ailerons de grille ont été utilisés, si le noyau est réutilisé, si des jambes ont été utilisées, la plateforme d'atterrissage utilisée, le bloc du noyau qui est un numéro utilisé pour séparer les versions de noyaux, le nombre de fois où ce noyau spécifique a été réutilisé et le numéro de série du noyau.

In [25]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API with the following URL:

Commençons maintenant à demander des données de lancement de fusée à l'API de SpaceX avec l'URL suivante :

In [26]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [27]:
response = requests.get(spacex_url)

Check the content of the response

Vérifiez le contenu de la réponse

In [28]:
print(response.content)

b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n\n<title>spacexdata.com | 525: SSL handshake failed</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />\n</head>\n<body>\n<div id="cf-wrapper">\n    <div id="cf-error-details" class="p-0">\n        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">\n            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-bla

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.

Vous devriez constater que la réponse contient une quantité considérable d'informations sur les lancements de SpaceX. Essayons maintenant de trouver d'autres informations pertinentes pour ce projet.

### Task 1: Request and parse the SpaceX launch data using the GET request

Tâche 1 : Demander et analyser les données de lancement de SpaceX à l’aide de la requête GET.

To make the requested JSON results more consistent, we will use the following static response object for this project:

Pour rendre les résultats JSON demandés plus cohérents, nous utiliserons l'objet de réponse statique suivant pour ce projet :

In [29]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code

Nous devrions constater que la requête a abouti avec le code de réponse 200.

In [30]:
response=requests.get(static_json_url)

In [31]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>

Nous décodons maintenant le contenu de la réponse au format JSON à l'aide de la méthode `.json()` et le transformons en un dataframe Pandas à l'aide de la méthode `.json_normalize()`.

In [32]:
# Use json_normalize meethod to convert the json result into a dataframe
data = response.json() # Décodage du contenu de la réponse
df = pd.json_normalize(data)


Using the dataframe <code>data</code> print the first 5 rows


In [33]:
# Get the head of the dataframe
print(df.head())

       static_fire_date_utc  static_fire_date_unix    tbd    net  window  \
0  2006-03-17T00:00:00.000Z           1.142554e+09  False  False     0.0   
1                       NaN                    NaN  False  False     0.0   
2                       NaN                    NaN  False  False     0.0   
3  2008-09-20T00:00:00.000Z           1.221869e+09  False  False     0.0   
4                       NaN                    NaN  False  False     0.0   

                     rocket  success  \
0  5e9d0d95eda69955f709d1eb    False   
1  5e9d0d95eda69955f709d1eb    False   
2  5e9d0d95eda69955f709d1eb    False   
3  5e9d0d95eda69955f709d1eb     True   
4  5e9d0d95eda69955f709d1eb     True   

                                                                                                                                                                                details  \
0                                                                                                                  

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.

Vous remarquerez que de nombreuses données sont des identifiants. Par exemple, la colonne « fusée » ne contient aucune information sur la fusée, seulement un numéro d'identification.

Nous allons maintenant utiliser à nouveau l'API pour obtenir des informations sur les lancements à l'aide des identifiants attribués à chaque lancement. Plus précisément, nous utiliserons les colonnes « fusée », « charges utiles », « plateforme de lancement » et « noyaux ».

In [34]:
#import pandas as pd

# 1. Assurez-vous que 'data' est converti en DataFrame
data = pd.DataFrame(data)

# 2. Prenez le sous-ensemble des colonnes que vous voulez
features = ['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']
data_subset = data[features].copy()

# 3. Ensuite, appliquez votre filtre (ex: exactement 1 élément dans 'cores' et 'payloads')
data_filtered = data_subset[
    (data_subset['cores'].apply(len) == 1) & 
    (data_subset['payloads'].apply(len) == 1)
]
# We also want to convert the date_utc to a datetime datatype and then extracting the date leaving the time
data['date'] = pd.to_datetime(data['date_utc']).dt.date

#Using the date we will restrict the dates of the launches
data = data[data['date'] <= datetime.date(2020, 11, 13)]

In [35]:

# Lets take a subset of our dataframe keeping only the features we want and the flight number, and date_utc.
#data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters and rows that have multiple payloads in a single rocket.
#data = data[data['cores'].map(len) == 1]
#data = data[data['payloads'].map(len) == 1]

# Since payloads and cores are lists of size 1 we will also extract the single value in the list and replace the feature.
#data['cores'] = data['cores'].map(lambda x : x[0])
#data['payloads'] = data['payloads'].map(lambda x : x[0])

# We also want to convert the date_utc to a datetime datatype and then extracting the date leaving the time
#data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Using the date we will restrict the dates of the launches
#data = data[data['date'] <= datetime.date(2020, 11, 13)]

* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.

Concernant la fusée, nous souhaitons obtenir le nom du propulseur.

Concernant la charge utile, nous souhaitons obtenir sa masse et son orbite cible.

Concernant le pas de tir, nous souhaitons obtenir le nom du site de lancement utilisé, sa longitude et sa latitude.

Concernant les modules, nous souhaitons obtenir le résultat de l'atterrissage, son type, le nombre de vols effectués avec ce module, l'utilisation ou non de grilles d'atterrissage, sa réutilisation, l'utilisation ou non de jambes d'atterrissage, le pas de tir utilisé, le numéro de bloc du module (identifiable par un numéro permettant de distinguer les différentes versions), le nombre de réutilisations de ce module et son numéro de série.

Les données issues de ces requêtes seront stockées dans des listes et serviront à créer un nouveau dataframe.

In [36]:
#Global variables 
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:

Ces fonctions appliqueront globalement les résultats aux variables ci-dessus. Prenons l'exemple de la variable BoosterVersion. Avant d'appliquer getBoosterVersion, la liste est vide :

In [37]:
BoosterVersion

[]

Now, let's apply <code> getBoosterVersion</code> function method to get the booster version

Appliquons maintenant la méthode de fonction getBoosterVersion pour obtenir la version du booster.

In [40]:
#import requests

def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            # Complete your URL here (e.g., 'https://spacexdata.com' + str(x))
            url = f"https://spacexdata.com{x}" 
            response = requests.get(url)
            
            # Check if the request was successful
            if response.status_code == 200:
                try:
                    # Attempt to parse JSON
                    BoosterVersion.append(response.json()['name'])
                except requests.exceptions.JSONDecodeError:
                    print(f"Error decoding JSON for {x}. Response text: {response.text}")
                    BoosterVersion.append(None)
            else:
                print(f"Failed to retrieve data for {x}. Status code: {response.status_code}")
                BoosterVersion.append(None)



the list has now been update 

La liste a été mise à jour.

In [43]:
# Take a look at the first 5 elements to verify
BoosterVersion[0:5]

[]

we can apply the rest of the functions here:

nous pouvons appliquer le reste des fonctions ici :


In [45]:
#import requests

def getLaunchSite(data):
    for x in data.get('launchpad', []):
        if x:
            url = f"https://api.spacexdata.com/v4/launchpads/{str(x)}"
            response = requests.get(url)
            
            if response.ok: # Checks if HTTP status code is 200-299
                launchpad_data = response.json()
                # Extract and use your data here, e.g.:
                # longitude = launchpad_data['longitude']
                # latitude = launchpad_data['latitude']
                print(f"Success for pad {x}")
            else:
                print(f"Failed to retrieve data for {x}. Status: {response.status_code}")


In [46]:
# Call getLaunchSite
#getLaunchSite(data)

In [49]:
for load_item in data['payloads']:
    # Ensure it's not empty/null
    if load_item:
        # Check if the payload is a list (nested payload info) or a single payload ID string
        if isinstance(load_item, list):
            for payload_id in load_item:
                url = f"https://api.spacexdata.com/v4/payloads/{payload_id}"
                response = requests.get(url)
                
                # Verify the request was successful before parsing JSON
                if response.status_code == 200:
                    try:
                        payload_data = response.json()
                        PayloadMass.append(payload_data.get('mass_kg'))
                        Orbit.append(payload_data.get('orbit'))
                    except ValueError:
                        print(f"Failed to decode JSON for payload ID: {payload_id}")
                else:
                    print(f"API Error {response.status_code} for payload ID: {payload_id}")
                    PayloadMass.append(None)
                    Orbit.append(None)
        else:
            # If load_item is already the payload ID string
            url = f"https://api.spacexdata.com/v4/payloads/{load_item}"
            response = requests.get(url)
            
            if response.status_code == 200:
                try:
                    payload_data = response.json()
                    PayloadMass.append(payload_data.get('mass_kg'))
                    Orbit.append(payload_data.get('orbit'))
                except ValueError:
                    print(f"Failed to decode JSON for payload ID: {load_item}")
            else:
                print(f"API Error {response.status_code} for payload ID: {load_item}")
                PayloadMass.append(None)
                Orbit.append(None)


API Error 525 for payload ID: 5eb0e4b5b6c3bb0006eeb1e1
API Error 525 for payload ID: 5eb0e4b6b6c3bb0006eeb1e2
API Error 525 for payload ID: 5eb0e4b6b6c3bb0006eeb1e3
API Error 525 for payload ID: 5eb0e4b6b6c3bb0006eeb1e4
API Error 525 for payload ID: 5eb0e4b7b6c3bb0006eeb1e5
API Error 525 for payload ID: 5eb0e4b7b6c3bb0006eeb1e6
API Error 525 for payload ID: 5eb0e4b7b6c3bb0006eeb1e7
API Error 525 for payload ID: 5eb0e4b9b6c3bb0006eeb1e8
API Error 525 for payload ID: 5eb0e4b9b6c3bb0006eeb1e9
API Error 525 for payload ID: 5eb0e4bab6c3bb0006eeb1ea
API Error 525 for payload ID: 5eb0e4bab6c3bb0006eeb1eb
API Error 525 for payload ID: 5eb0e4bab6c3bb0006eeb1ec
API Error 525 for payload ID: 5eb0e4bbb6c3bb0006eeb1ed
API Error 525 for payload ID: 5eb0e4bbb6c3bb0006eeb1ee
API Error 525 for payload ID: 5eb0e4bbb6c3bb0006eeb1ef
API Error 525 for payload ID: 5eb0e4bbb6c3bb0006eeb1f0
API Error 525 for payload ID: 5eb0e4bbb6c3bb0006eeb1f1
API Error 525 for payload ID: 5eb0e4bcb6c3bb0006eeb1f2
API Error 

In [50]:
# Call getPayloadData
#getPayloadData(data)

In [54]:
#import requests

def getCoreData(data, Block):
    for core in data['cores']:
        # Vérifie si core n'est pas None (l'identifiant existe)
        if core is not None:
            # On utilise directement core, car c'est déjà l'identifiant (ex: "B1051")
            url = f"https://api.spacexdata.com/v4/cores/{core}"
            response = requests.get(url).json()
            Block.append(response['block'])

# Exemple d'initialisation de votre liste
# Block = []
# getCoreData(data, Block)


In [52]:
# Call getCoreData
#getCoreData(data)

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.

Enfin, construisons notre ensemble de données à partir des données obtenues. Nous combinons les colonnes en un dictionnaire.

In [55]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}


Then, we need to create a Pandas data frame from the dictionary launch_dict.

Ensuite, nous devons créer un dataframe Pandas à partir du dictionnaire launch dict.

In [57]:
#import pandas as pd

# Correction : utiliser json_normalize si les longueurs ou les imbrications diffèrent
df = pd.json_normalize(launch_dict)

# Display the first 5 rows to verify your data
print(df.head())


                                                                                                                                                                                                                                                                                                                                                                                                    FlightNumber  \
0  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, ...]   

                                                                                                                                                                                               

In [58]:
# Create a data from launch_dict
# Create a DataFrame from the launch_dict
#df = pd.DataFrame.from_dict(launch_dict)

# Display the first 5 rows to verify your data
#print(df.head())

Show the summary of the dataframe


In [59]:
# Show the head of the dataframe

# 1. Display the structural summary (data types, non-null counts)
print("--- DataFrame Information ---")
print(df.info())

# 2. Display the statistical summary (mean, min, max, percentiles for numerical columns)
print("\n--- Statistical Summary ---")
print(df.describe())


--- DataFrame Information ---
<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   FlightNumber    1 non-null      object
 1   Date            1 non-null      object
 2   BoosterVersion  1 non-null      object
 3   PayloadMass     1 non-null      object
 4   Orbit           1 non-null      object
 5   LaunchSite      1 non-null      object
 6   Outcome         1 non-null      object
 7   Flights         1 non-null      object
 8   GridFins        1 non-null      object
 9   Reused          1 non-null      object
 10  Legs            1 non-null      object
 11  LandingPad      1 non-null      object
 12  Block           1 non-null      object
 13  ReusedCount     1 non-null      object
 14  Serial          1 non-null      object
 15  Longitude       1 non-null      object
 16  Latitude        1 non-null      object
dtypes: object(17)
memory usage: 268.0+ byte

### Task 2: Filter the dataframe to only include `Falcon 9` launches

Tâche 2 : Filtrer le dataframe pour n’inclure que les lancements de Falcon 9

Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.

Enfin, nous supprimerons les lancements de Falcon 1 pour ne conserver que ceux de Falcon 9. Filtrez le dataframe à l'aide de la colonne BoosterVersion afin de ne garder que les lancements de Falcon 9. Enregistrez les données filtrées dans un nouveau dataframe nommé data_falcon9.

In [60]:
# Hint data['BoosterVersion']!='Falcon 1'
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1']

Now that we have removed some values we should reset the FlgihtNumber column

Maintenant que nous avons supprimé certaines valeurs, nous devons réinitialiser la colonne FlightNumber.

In [61]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,"[2006-03-24, 2007-03-21, 2008-08-03, 2008-09-28, 2009-07-13, 2010-06-04, 2010-12-08, 2012-05-22, 2012-10-08, 2013-03-01, 2013-09-29, 2013-12-03, 2014-01-06, 2014-04-18, 2014-07-14, 2014-08-05, 2014-09-07, 2014-09-21, 2015-01-10, 2015-02-11, 2015-03-02, 2015-04-14, 2015-04-27, 2015-06-28, 2015-12-22, 2016-01-17, 2016-03-04, 2016-04-08, 2016-05-06, 2016-05-27, 2016-06-15, 2016-07-18, 2016-08-14, 2016-09-01, 2017-01-14, 2017-02-19, 2017-03-16, 2017-03-30, 2017-05-01, 2017-05-15, 2017-06-03, 2017-06-23, 2017-06-25, 2017-07-05, 2017-08-14, 2017-08-24, 2017-09-07, 2017-10-09, 2017-10-11, 2017-10-30, 2017-12-15, 2017-12-23, 2018-01-08, 2018-01-31, 2018-02-06, 2018-02-22, 2018-03-06, 2018-03-30, 2018-04-02, 2018-04-18, 2018-05-11, 2018-05-22, 2018-06-04, 2018-06-29, 2018-07-22, 2018-07-25, 2018-08-07, 2018-09-10, 2018-10-08, 2018-11-15, 2018-12-03, 2018-12-05, 2018-12-23, 2019-01-11, 2019-02-22, 2019-03-02, 2019-04-11, 2019-05-04, 2019-05-24, 2019-06-12, 2019-06-25, 2019-07-25, 2019-08-06, 2019-11-11, 2019-12-05, 2019-12-17, 2020-01-07, 2020-01-19, 2020-01-29, 2020-02-17, 2020-03-07, 2020-03-18, 2020-04-22, 2020-05-30, 2020-06-04, 2020-06-13, 2020-06-30, 2020-07-20, 2020-08-07, 2020-08-18, ...]",[],"[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, ...]","[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, ...]",[],[],[],[],[],[],[],[],[],[],[],[]


## Data Wrangling

Préparation des données

We can see below that some of the rows are missing values in our dataset.

On peut constater ci-dessous que certaines lignes de notre ensemble de données présentent des valeurs manquantes.

In [74]:
# Afficher le décompte des valeurs manquantes par colonne
print(data_falcon9.isnull().sum())


FlightNumber      0
Date              0
BoosterVersion    0
PayloadMass       0
Orbit             0
LaunchSite        0
Outcome           0
Flights           0
GridFins          0
Reused            0
Legs              0
LandingPad        0
Block             0
ReusedCount       0
Serial            0
Longitude         0
Latitude          0
dtype: int64


In [75]:
#data_falcon9.isnull()

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.

Avant de poursuivre, il nous faut traiter ces valeurs manquantes. La colonne LandingPad conservera la valeur « None » pour indiquer que les plateformes d'atterrissage n'ont pas été utilisées.

### Task 3: Dealing with Missing Values

Tâche 3 : Traitement des valeurs manquantes

Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.

Calculez ci-dessous la moyenne de la masse de la charge utile à l'aide de la fonction `.mean()`. Utilisez ensuite cette moyenne et la fonction `.replace()` pour remplacer les valeurs `np.nan` dans les données par la moyenne calculée.

In [67]:
print(data_falcon9.columns)

Index(['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit',
       'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs',
       'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude',
       'Latitude'],
      dtype='str')


In [79]:
import numpy as np
import pandas as pd

# 1. Convertir la colonne en valeurs numériques (les erreurs deviennent NaN)
data_falcon9['PayloadMass'] = pd.to_numeric(data_falcon9['PayloadMass'], errors='coerce')

# 2. Calculer la moyenne de la colonne PayloadMass
payload_mass_mean = data_falcon9['PayloadMass'].mean()

# 3. Remplacer les valeurs NaN par la moyenne calculée
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(payload_mass_mean)


You should see the number of missing values of the <code>PayLoadMass</code> change to zero.

Le nombre de valeurs manquantes de PayLoadMass devrait passer à zéro.

Now we should have no missing values in our dataset except for in <code>LandingPad</code>.

Nous ne devrions plus avoir de valeurs manquantes dans notre ensemble de données, sauf dans LandingPad.

We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 

Nous pouvons maintenant l'exporter au format CSV pour la section suivante, mais pour assurer la cohérence des réponses, dans le prochain TP, nous fournirons des données sur une plage de dates prédéfinie.

<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
